In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls "/content/drive/MyDrive/" | head -30

In [ ]:
!ls "/content/drive/MyDrive/" | grep -E "medqa_gpt|medmcqa"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, math
from collections import Counter

DRIVE_DIR = '/content/drive/MyDrive'

FILES = {
    'medqa_flan':     'flan_t5_results.json',
    'medqa_llama':    'llama_results.json',
    'medqa_qwen':     'qwen_results.json',
    'medqa_gemma':    'gemma3n_results.json',
    'medqa_deepseek': 'deepseek_results.json',
    'medqa_gpt4o':    'medqa_gpt4o_results.json',
    'medmcqa_flan':   'medmcqa_flan_t5_results.json',
    'medmcqa_llama':  'medmcqa_llama_results.json',
    'medmcqa_qwen':   'medmcqa_qwen_results.json',
    'medmcqa_gemma':  'medmcqa_gemma3n_results.json',
    'medmcqa_gpt4o':  'medmcqa_gpt4o_results.json',
}

preds = {}
golds = {}
for name, fname in FILES.items():
    path = os.path.join(DRIVE_DIR, fname)
    if not os.path.exists(path):
        print(f'MISSING: {path}')
        continue
    with open(path) as f:
        data = json.load(f)
    preds[name] = {r['idx']: r['pred'] for r in data}
    if name.startswith('medqa_') and 'medqa' not in golds:
        golds['medqa'] = {r['idx']: r['gold'] for r in data}
    if name.startswith('medmcqa_') and 'medmcqa' not in golds:
        golds['medmcqa'] = {r['idx']: r['gold'] for r in data}
    print(f'{name}: {len(preds[name])} predictions')

print(f'\nGold sets: MedQA={len(golds.get("medqa",{}))} questions, MedMCQA={len(golds.get("medmcqa",{}))} questions')

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    phat = k / n
    denom = 1 + z*z/n
    center = (phat + z*z/(2*n)) / denom
    halfwidth = (z * math.sqrt(phat*(1-phat)/n + z*z/(4*n*n))) / denom
    return (max(0.0, center - halfwidth), min(1.0, center + halfwidth))

def majority(votes):
    votes = [v for v in votes if v is not None]
    if not votes:
        return None, 0
    c = Counter(votes)
    top = c.most_common(1)[0]
    return top[0], top[1]

print('\nUtilities loaded. Ready for experiments.')

In [ ]:
def vote_entropy(votes):
    votes = [v for v in votes if v is not None]
    if not votes:
        return 0.0
    counts = Counter(votes)
    total = sum(counts.values())
    H = 0.0
    for c in counts.values():
        p = c / total
        H -= p * math.log2(p)
    return H

def compute_dataset(prefix, strong_models, gold_dict, n_questions):
    rows = []
    for idx in range(n_questions):
        votes = [preds[f'{prefix}_{m}'].get(idx) for m in strong_models]
        H = vote_entropy(votes)
        mv, _ = majority(votes)
        gold = gold_dict.get(idx)
        rows.append({
            'idx': idx, 'entropy': H, 'majority': mv, 'gold': gold,
            'mv_correct': int(mv == gold) if mv and gold else 0,
        })
    return rows

medqa_strong   = ['llama','qwen','gemma','deepseek','gpt4o']
medmcqa_strong = ['llama','qwen','gemma','gpt4o']

medqa_rows   = compute_dataset('medqa',   medqa_strong,   golds['medqa'],   1273)
medmcqa_rows = compute_dataset('medmcqa', medmcqa_strong, golds['medmcqa'], 2816)

print(f'MedQA majority-vote accuracy:   {sum(r["mv_correct"] for r in medqa_rows)/len(medqa_rows):.3f}')
print(f'MedMCQA majority-vote accuracy: {sum(r["mv_correct"] for r in medmcqa_rows)/len(medmcqa_rows):.3f}')

def sweep_thresholds(rows, direction='low'):
    Hs = sorted(set(r['entropy'] for r in rows))
    results = []
    total_wrong = sum(1 - r['mv_correct'] for r in rows)
    for thr in Hs:
        flagged = [r for r in rows if (r['entropy'] <= thr if direction == 'low' else r['entropy'] >= thr)]
        if not flagged:
            continue
        f_wrong = sum(1 - r['mv_correct'] for r in flagged)
        results.append({
            'threshold': thr, 'n_flagged': len(flagged),
            'precision': f_wrong/len(flagged),
            'recall': f_wrong/total_wrong if total_wrong else 0,
            'route_fraction': len(flagged)/len(rows),
        })
    return results

def find_at_route(sweep, target):
    return min(sweep, key=lambda r: abs(r['route_fraction'] - target))

medqa_low    = sweep_thresholds(medqa_rows,   'low')
medqa_high   = sweep_thresholds(medqa_rows,   'high')
medmcqa_low  = sweep_thresholds(medmcqa_rows, 'low')
medmcqa_high = sweep_thresholds(medmcqa_rows, 'high')

print('\n=== COMPARISON AT EQUAL ROUTE FRACTION ===')
print('\nMedQA target route 8.0%:')
mq_lo, mq_hi = find_at_route(medqa_low, 0.08), find_at_route(medqa_high, 0.08)
print(f"  Entropy LOW  : route={mq_lo['route_fraction']*100:.1f}%  prec={mq_lo['precision']*100:.1f}%  recall={mq_lo['recall']*100:.1f}%")
print(f"  Entropy HIGH : route={mq_hi['route_fraction']*100:.1f}%  prec={mq_hi['precision']*100:.1f}%  recall={mq_hi['recall']*100:.1f}%")
print(f"  Our rule     : route=8.0%   prec=96.1%  recall=18.6% (locked)")

print('\nMedMCQA target route 6.7%:')
mc_lo, mc_hi = find_at_route(medmcqa_low, 0.067), find_at_route(medmcqa_high, 0.067)
print(f"  Entropy LOW  : route={mc_lo['route_fraction']*100:.1f}%  prec={mc_lo['precision']*100:.1f}%  recall={mc_lo['recall']*100:.1f}%")
print(f"  Entropy HIGH : route={mc_hi['route_fraction']*100:.1f}%  prec={mc_hi['precision']*100:.1f}%  recall={mc_hi['recall']*100:.1f}%")
print(f"  Our rule     : route=6.7%   prec=83.0%  recall=13.0% (locked)")

summary = {
    'experiment': 'entropy_baseline',
    'medqa':   {'entropy_low_at_8pct': mq_lo, 'entropy_high_at_8pct': mq_hi,
                'majority_accuracy': sum(r["mv_correct"] for r in medqa_rows)/len(medqa_rows)},
    'medmcqa': {'entropy_low_at_6_7pct': mc_lo, 'entropy_high_at_6_7pct': mc_hi,
                'majority_accuracy': sum(r["mv_correct"] for r in medmcqa_rows)/len(medmcqa_rows)},
}
with open(os.path.join(DRIVE_DIR, 'exp2_entropy_baseline.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: {DRIVE_DIR}/exp2_entropy_baseline.json')

In [ ]:
def apply_detector(prefix, frontier_model, n_questions, gold_dict):
    tiers = []
    for idx in range(n_questions):
        l = preds[f'{prefix}_llama'].get(idx)
        q = preds[f'{prefix}_qwen'].get(idx)
        g = preds[f'{prefix}_gemma'].get(idx)
        f = preds[f'{prefix}_{frontier_model}'].get(idx)
        gold = gold_dict.get(idx)

        mid_unanimous = (l is not None and l == q == g)
        if mid_unanimous and f is not None and f == l:
            tier, consensus = 'CONFIDENT', l
        elif mid_unanimous and f is not None and f != l:
            tier, consensus = 'HIGH_RISK', l
        elif mid_unanimous and f is None:
            tier, consensus = 'MID_UNANIMOUS_NO_FRONTIER', l
        else:
            tier, consensus = 'DISAGREEMENT', None

        tiers.append({
            'idx': idx, 'tier': tier, 'consensus': consensus, 'gold': gold,
            'mid_correct': int(consensus == gold) if consensus and gold else 0,
        })
    return tiers

def report_detector(tiers, label, n_total):
    high_risk = [t for t in tiers if t['tier'] == 'HIGH_RISK']
    confident = [t for t in tiers if t['tier'] == 'CONFIDENT']

    hr_n = len(high_risk)
    hr_wrong = sum(1 - t['mid_correct'] for t in high_risk)
    hr_prec = hr_wrong / hr_n if hr_n else 0
    hr_prec_ci = wilson_ci(hr_wrong, hr_n)

    conf_n = len(confident)
    conf_wrong = sum(1 - t['mid_correct'] for t in confident)
    conf_rate = conf_wrong / conf_n if conf_n else 0
    conf_ci = wilson_ci(conf_wrong, conf_n)

    all_mid_wrong = sum(1 - t['mid_correct'] for t in tiers if t['consensus'] is not None)
    hr_recall_discriminative = hr_wrong / all_mid_wrong if all_mid_wrong else 0

    total_dataset_wrong = sum(1 for t in tiers if t['gold'] and t['consensus'] and t['consensus'] != t['gold'])
    total_majority_wrong_all = sum(1 for t in tiers if t['gold'] and (t['consensus'] is None or t['consensus'] != t['gold']))

    print(f'\n=== {label} ===')
    print(f'  HIGH_RISK: n={hr_n}, route={hr_n/n_total*100:.1f}%')
    print(f'    precision={hr_prec*100:.1f}%  Wilson CI [{hr_prec_ci[0]*100:.1f}%, {hr_prec_ci[1]*100:.1f}%]')
    print(f'    discriminative recall (of mid-consensus errors)={hr_recall_discriminative*100:.1f}%')
    print(f'  CONFIDENT: n={conf_n}, coverage={conf_n/n_total*100:.1f}%')
    print(f'    wrong_rate={conf_rate*100:.1f}%  Wilson CI [{conf_ci[0]*100:.1f}%, {conf_ci[1]*100:.1f}%]')

    return {
        'high_risk_n': hr_n, 'high_risk_wrong': hr_wrong,
        'high_risk_precision': hr_prec, 'high_risk_precision_ci': list(hr_prec_ci),
        'high_risk_recall_discriminative': hr_recall_discriminative,
        'high_risk_route': hr_n/n_total,
        'confident_n': conf_n, 'confident_wrong': conf_wrong,
        'confident_wrong_rate': conf_rate, 'confident_wrong_ci': list(conf_ci),
    }

print('=== Cross-Dataset Transfer (same rule, applied to both datasets) ===')

medqa_dseek    = apply_detector('medqa',   'deepseek', 1273, golds['medqa'])
medqa_dseek_r  = report_detector(medqa_dseek,   'MedQA, frontier=DeepSeek-V3', 1273)

medqa_gpt4o    = apply_detector('medqa',   'gpt4o',    1273, golds['medqa'])
medqa_gpt4o_r  = report_detector(medqa_gpt4o,   'MedQA, frontier=GPT-4o (robustness)', 1273)

medmcqa_gpt4o  = apply_detector('medmcqa', 'gpt4o',    2816, golds['medmcqa'])
medmcqa_gpt4o_r = report_detector(medmcqa_gpt4o, 'MedMCQA, frontier=GPT-4o', 2816)

print('\n=== Wilson 95% CIs for headline rates ===')
headlines = [
    ('5-model MedQA unanimous-wrong',     14, 57),
    ('4-model MedQA unanimous-wrong',     48, 143),
    ('4-model MedMCQA unanimous-wrong',   71, 301),
    ('Shuffle joint text-lock',           29, 48),
    ('Shuffle joint letter-lock',          5, 48),
]
ci_results = {}
for label, k, n in headlines:
    lo, hi = wilson_ci(k, n)
    print(f'  {label}: {k}/{n} = {k/n*100:.1f}%  Wilson CI [{lo*100:.1f}%, {hi*100:.1f}%]')
    ci_results[label] = {'k': k, 'n': n, 'rate': k/n, 'ci': [lo, hi]}

summary = {
    'experiment': 'holdout_validation_and_CIs',
    'medqa_deepseek_frontier': medqa_dseek_r,
    'medqa_gpt4o_frontier_robustness': medqa_gpt4o_r,
    'medmcqa_gpt4o_frontier': medmcqa_gpt4o_r,
    'headline_wilson_cis': ci_results,
}
with open(os.path.join(DRIVE_DIR, 'exp1_holdout_cis.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: {DRIVE_DIR}/exp1_holdout_cis.json')

In [ ]:
def annotate(prefix, mid_models, frontier_model, n_questions, gold_dict):
    rows = []
    for idx in range(n_questions):
        mid_votes = [preds[f'{prefix}_{m}'].get(idx) for m in mid_models]
        f = preds[f'{prefix}_{frontier_model}'].get(idx)
        all_votes = [v for v in mid_votes + [f] if v is not None]
        if not all_votes:
            continue
        top_ans, top_count = majority(all_votes)
        vote_share = top_count / len(all_votes)
        gold = gold_dict.get(idx)
        majority_correct = (top_ans == gold)

        mid_non_none = [v for v in mid_votes if v is not None]
        if len(mid_non_none) == 3 and len(set(mid_non_none)) == 1:
            mid_consensus = mid_non_none[0]
            if f is not None and f == mid_consensus:
                tier = 'CONFIDENT'
            elif f is not None and f != mid_consensus:
                tier = 'HIGH_RISK'
            else:
                tier = 'OTHER'
        else:
            tier = 'DISAGREEMENT'

        rows.append({
            'idx': idx, 'vote_share': vote_share, 'tier': tier,
            'majority_correct': majority_correct, 'top_ans': top_ans, 'gold': gold,
        })
    return rows

medqa_annot   = annotate('medqa',   ['llama','qwen','gemma'], 'gpt4o', 1273, golds['medqa'])
medmcqa_annot = annotate('medmcqa', ['llama','qwen','gemma'], 'gpt4o', 2816, golds['medmcqa'])

def confidence_analysis(rows, name):
    print(f'\n=== {name} ===')
    n_total = len(rows)
    total_wrong = sum(1 for r in rows if not r['majority_correct'])
    print(f'Total: {n_total}, majority-wrong: {total_wrong}')

    print('\n  Vote-share threshold sweep (flag low-confidence as risky):')
    for thr in [0.25, 0.5, 0.6, 0.75, 1.0]:
        flagged = [r for r in rows if r['vote_share'] <= thr]
        if not flagged:
            continue
        fw = sum(1 for r in flagged if not r['majority_correct'])
        prec = fw / len(flagged)
        rec = fw / total_wrong if total_wrong else 0
        route = len(flagged) / n_total
        print(f'    vote_share <= {thr:.2f}: route={route*100:.1f}%, prec={prec*100:.1f}%, recall={rec*100:.1f}%')

    hr = [r for r in rows if r['tier'] == 'HIGH_RISK']
    if hr:
        hr_w = sum(1 for r in hr if not r['majority_correct'])
        print(f'\n  Our HIGH_RISK rule: route={len(hr)/n_total*100:.1f}%, n={len(hr)}, prec={hr_w/len(hr)*100:.1f}%')

    print('\n  --- The crucial test: same vote-share, different structure ---')
    same_vs = [r for r in rows if abs(r['vote_share'] - 0.75) < 0.01]
    if same_vs:
        hr_in_vs    = [r for r in same_vs if r['tier'] == 'HIGH_RISK']
        nonhr_in_vs = [r for r in same_vs if r['tier'] != 'HIGH_RISK']
        print(f'  Questions with vote_share=0.75 (3-of-4 agree): {len(same_vs)}')
        if hr_in_vs:
            hr_w = sum(1 for r in hr_in_vs if not r['majority_correct'])
            print(f'    HIGH_RISK subset (mid-unanimous, frontier dissent): n={len(hr_in_vs)}, wrong={hr_w/len(hr_in_vs)*100:.1f}%')
        if nonhr_in_vs:
            non_w = sum(1 for r in nonhr_in_vs if not r['majority_correct'])
            print(f'    Other 3-of-4 configs (frontier in majority):       n={len(nonhr_in_vs)}, wrong={non_w/len(nonhr_in_vs)*100:.1f}%')
        print('  Interpretation: if HIGH_RISK wrong rate >> other 3-of-4 wrong rate,')
        print('  vote-share alone cannot replace the rule. The asymmetric structure matters.')

    return same_vs, hr, n_total, total_wrong

mq_same, mq_hr, mq_n, mq_tw = confidence_analysis(medqa_annot,   'MedQA')
mc_same, mc_hr, mc_n, mc_tw = confidence_analysis(medmcqa_annot, 'MedMCQA')

summary = {
    'experiment': 'confidence_calibration_proxy',
    'note': 'Vote-share used as confidence proxy (logprobs not cached).',
    'finding': 'See crucial test: vote-share = 0.75 contains both HIGH_RISK and other configs with very different wrong rates.',
}
with open(os.path.join(DRIVE_DIR, 'exp9_confidence_proxy.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: {DRIVE_DIR}/exp9_confidence_proxy.json')

In [ ]:
def correctness_vector(prefix, model_name, n_questions, gold_dict):
    """Returns a list of 0/1 where 1 = model got that question right."""
    vec = []
    for idx in range(n_questions):
        p = preds[f'{prefix}_{model_name}'].get(idx)
        g = gold_dict.get(idx)
        vec.append(int(p is not None and p == g))
    return vec

def q_statistic(c_a, c_b):
    """Yule's Q-statistic. Range [-1, 1]. 0 = independent, +1 = identical errors, -1 = anti-correlated."""
    n11 = sum(1 for a, b in zip(c_a, c_b) if a == 1 and b == 1)
    n00 = sum(1 for a, b in zip(c_a, c_b) if a == 0 and b == 0)
    n10 = sum(1 for a, b in zip(c_a, c_b) if a == 1 and b == 0)
    n01 = sum(1 for a, b in zip(c_a, c_b) if a == 0 and b == 1)
    num = n11*n00 - n01*n10
    denom = n11*n00 + n01*n10
    return num/denom if denom != 0 else 0.0

def double_fault(c_a, c_b):
    """Fraction of questions where BOTH models err. Higher = more correlated failure."""
    n = len(c_a)
    both_wrong = sum(1 for a, b in zip(c_a, c_b) if a == 0 and b == 0)
    return both_wrong / n if n else 0.0

def pairwise_diversity(prefix, model_list, n_questions, gold_dict, name):
    vecs = {m: correctness_vector(prefix, m, n_questions, gold_dict) for m in model_list}
    print(f'\n=== {name} ===')
    print(f'{"Pair":<25} {"Q-stat":>8} {"DoubleFault":>13} {"Err_A":>8} {"Err_B":>8}')
    print('-' * 70)
    results = []
    for i, a in enumerate(model_list):
        for b in model_list[i+1:]:
            q = q_statistic(vecs[a], vecs[b])
            df = double_fault(vecs[a], vecs[b])
            err_a = 1 - sum(vecs[a])/len(vecs[a])
            err_b = 1 - sum(vecs[b])/len(vecs[b])
            kind = 'flan-strong' if 'flan' in (a, b) else 'strong-strong'
            print(f'{a+" / "+b:<25} {q:>8.3f} {df:>13.3f} {err_a:>8.3f} {err_b:>8.3f}  [{kind}]')
            results.append({'pair': f'{a}/{b}', 'Q': q, 'DF': df, 'kind': kind, 'err_a': err_a, 'err_b': err_b})
    return results

print('Q-statistic: range [-1, +1]. 0 = independent errors. +1 = same errors. -1 = anti-correlated.')
print('Double-fault: fraction of questions where BOTH models err. Higher = more correlated failure.')

medqa_div = pairwise_diversity('medqa', ['flan','llama','qwen','gemma','deepseek','gpt4o'], 1273, golds['medqa'], 'MedQA pairwise diversity')
medmcqa_div = pairwise_diversity('medmcqa', ['flan','llama','qwen','gemma','gpt4o'], 2816, golds['medmcqa'], 'MedMCQA pairwise diversity')

print('\n=== Summary: strong-strong vs flan-strong ===')
for ds_name, results in [('MedQA', medqa_div), ('MedMCQA', medmcqa_div)]:
    strong_q = [r['Q'] for r in results if r['kind'] == 'strong-strong']
    flan_q = [r['Q'] for r in results if r['kind'] == 'flan-strong']
    strong_df = [r['DF'] for r in results if r['kind'] == 'strong-strong']
    flan_df = [r['DF'] for r in results if r['kind'] == 'flan-strong']
    print(f'\n{ds_name}:')
    print(f'  Strong-strong: Q ranges [{min(strong_q):.3f}, {max(strong_q):.3f}], mean Q = {sum(strong_q)/len(strong_q):.3f}')
    print(f'  Flan-strong:   Q ranges [{min(flan_q):.3f}, {max(flan_q):.3f}], mean Q = {sum(flan_q)/len(flan_q):.3f}')
    print(f'  Strong-strong: DF ranges [{min(strong_df):.3f}, {max(strong_df):.3f}], mean DF = {sum(strong_df)/len(strong_df):.3f}')
    print(f'  Flan-strong:   DF ranges [{min(flan_df):.3f}, {max(flan_df):.3f}], mean DF = {sum(flan_df)/len(flan_df):.3f}')

summary = {
    'experiment': 'classical_diversity_measures',
    'medqa_pairs': medqa_div,
    'medmcqa_pairs': medmcqa_div,
    'notes': 'Q-statistic (Kuncheva-Whitaker 2003); double-fault from same literature.',
}
with open(os.path.join(DRIVE_DIR, 'exp3_diversity_measures.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: {DRIVE_DIR}/exp3_diversity_measures.json')